# 01 Corpus Inspection and Ingestion

Kernel: **Python (NLP)**. This notebook uses only the frozen handoff files and saves derived outputs in the Phase 2 directories.

## Scope

Verify the frozen corpus, construct one deterministic title-plus-content retrieval text per source document, inspect the pinned MiniLM tokenizer/model limit, and report all potentially truncated documents. This notebook does not use queries, answers, qrels, or test data.

In [1]:
from pathlib import Path
import os, json, hashlib, platform, unicodedata, re, time
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
import sentence_transformers, transformers, torch

# Find the handoff root from the notebook working directory instead of hardcoding this machine's path.
HANDOFF_ROOT = Path.cwd().resolve()
while not (HANDOFF_ROOT / "02_FROZEN_DATA").is_dir():
    if HANDOFF_ROOT.parent == HANDOFF_ROOT:
        raise FileNotFoundError("Could not locate handoff folder containing 02_FROZEN_DATA")
    HANDOFF_ROOT = HANDOFF_ROOT.parent
PHASE_ROOT = HANDOFF_ROOT / "Phase2"
BUILD_ROOT = PHASE_ROOT / "DocumentBuilding"
DATA_DIR = BUILD_ROOT / "data"
MANIFEST_DIR = BUILD_ROOT / "manifests"
LOG_DIR = BUILD_ROOT / "logs"
for folder in (DATA_DIR, MANIFEST_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Keep the Hugging Face cache within the requested shared NLP environment folder.
HF_HOME = HANDOFF_ROOT / "NLP" / "hf_cache"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
DEVICE = "cpu"  # PyTorch reported MPS unavailable during the environment check.
CORPUS_PATH = HANDOFF_ROOT / "02_FROZEN_DATA" / "Corpus_4192_Gaming_Documents.csv"
print("Handoff:", HANDOFF_ROOT)
print("Model:", MODEL_ID, "revision:", MODEL_REVISION, "device:", DEVICE)

/Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 15:27:17.243000 8027 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Handoff: /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1
Model: sentence-transformers/all-MiniLM-L6-v2 revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 device: cpu


In [2]:
# Read only the corpus fields required for document retrieval; ignore unrelated metadata columns.
source_header = pd.read_csv(CORPUS_PATH, nrows=0).columns.tolist()
required_columns = {"id", "title", "contents", "game_name"}
assert required_columns.issubset(source_header), f"Unexpected corpus schema: {source_header}"
corpus = pd.read_csv(CORPUS_PATH, usecols=["game_name", "id", "title", "contents"], dtype={"id":"string"})
print("Rows:", len(corpus), "columns read:", corpus.columns.tolist())
print("Missing values:", corpus[["id", "title", "contents", "game_name"]].isna().sum())
print("Duplicate IDs:", int(corpus["id"].duplicated().sum()))
assert len(corpus) == 4192
assert corpus["id"].notna().all() and corpus["id"].is_unique
assert corpus[["title", "contents", "game_name"]].notna().all().all()
assert corpus["id"].astype(str).str.len().gt(0).all()

Rows: 4192 columns read: ['game_name', 'id', 'title', 'contents']
Missing values: id           0
title        0
contents     0
game_name    0
dtype: int64
Duplicate IDs: 0


In [3]:
# Apply only deterministic Unicode NFC and newline normalization, preserving wording and whitespace otherwise.
def normalize_newlines(value):
    text = unicodedata.normalize("NFC", str(value))
    return text.replace("\r\n", "\n").replace("\r", "\n")

def make_document_text(title, contents):
    return f"Title: {normalize_newlines(title)}\nContent: {normalize_newlines(contents)}"

corpus["document_text"] = [make_document_text(t, c) for t, c in zip(corpus["title"], corpus["contents"])]
corpus["document_text_sha256"] = corpus["document_text"].map(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
corpus["document_chars"] = corpus["document_text"].str.len()
print("Document representation:", len(corpus), "one document per source row")
print(corpus[["id", "title", "document_chars"]].head(3).to_string(index=False))
print("Sample text (first 600 characters):\n", corpus.loc[0, "document_text"][:600])

Document representation: 4192 one document per source row
                                  id                                             title  document_chars
fa927758-3f31-4b93-be4a-8ef9810b4e81 2025-06-10-Dune: Awakening – 1.1.0.11 Patch Notes            1112
3d2f63ae-368c-4914-b7e3-9dbc6a0885a7            2025-06-10-Dune: Awakening is Out Now!            2672
f01be39a-f8b3-4cbb-935f-9e3a8f630720  2025-06-06-Dune: Awakening – 1.1.0.5 Patch Notes            1349
Sample text (first 600 characters):
 Title: 2025-06-10-Dune: Awakening – 1.1.0.11 Patch Notes
Content: # Dune: Awakening – 1.1.0.11 Patch Notes

## Publication Date

### 2025-06-10

---

June 10

**Dune: Awakening – 1.1.0.11 Patch Notes**

Greetings, Sleepers,

Several updates and fixes have been deployed to enhance server stability and address in-game issues.

This hotfix requires a client-side download. Players are advised to restart their Steam client if the download does not appear automatically. Servers will be coming b

In [4]:
# Download only the pinned Sentence Transformers assets needed for tokenization/encoding.
# The full revision is locked by its immutable commit hash; optional ONNX/OpenVINO exports are omitted.
snapshot_dir = Path(snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(HF_HOME),
    allow_patterns=["*.json", "*.txt", "*.safetensors"],
))
assert snapshot_dir.name == MODEL_REVISION, f"Loaded snapshot is {snapshot_dir.name}, expected {MODEL_REVISION}"
model = SentenceTransformer(str(snapshot_dir), device=DEVICE)
model_commit = getattr(model[0].auto_model.config, "_commit_hash", None)
print("Snapshot directory:", snapshot_dir)
print("Model config commit:", model_commit)
print("SentenceTransformer max_seq_length:", model.max_seq_length)
print("Tokenizer model_max_length:", model.tokenizer.model_max_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Versions:", {"sentence-transformers":sentence_transformers.__version__, "transformers":transformers.__version__, "torch":torch.__version__})
assert model.max_seq_length > 0 and model.get_sentence_embedding_dimension() == 384
if model_commit is not None:
    assert model_commit == MODEL_REVISION

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files:   9%|▉         | 1/11 [00:00<00:02,  3.49it/s]

Fetching 11 files:  18%|█▊        | 2/11 [00:00<00:02,  3.57it/s]

Fetching 11 files:  45%|████▌     | 5/11 [00:00<00:00,  8.73it/s]

Fetching 11 files:  82%|████████▏ | 9/11 [00:00<00:00, 13.87it/s]

...

Fetching 11 files: 100%|██████████| 11/11 [00:11<00:00,  1.07s/it]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19124.10it/s]

Snapshot directory: /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/NLP/hf_cache/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Model config commit: None
SentenceTransformer max_seq_length: 256
Tokenizer model_max_length: 256
Embedding dimension: 384
Versions: {'sentence-transformers': '6.2.0', 'transformers': '5.21.0', 'torch': '2.14.1'}



/var/folders/6x/b5gh_pcd4pbb668mxxxjshjw0000gn/T/ipykernel_8027/444785067.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())
/var/folders/6x/b5gh_pcd4pbb668mxxxjshjw0000gn/T/ipykernel_8027/444785067.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.max_seq_length > 0 and model.get_sentence_embedding_dimension() == 384


In [5]:
# Measure token lengths without truncation, then compare against the model's actual SentenceTransformer limit.
tokenizer = model.tokenizer
MAX_SEQUENCE_LENGTH = int(model.max_seq_length)
token_lengths = []
BATCH_SIZE = 64
for start in range(0, len(corpus), BATCH_SIZE):
    batch_text = corpus["document_text"].iloc[start:start+BATCH_SIZE].tolist()
    tokenized = tokenizer(batch_text, add_special_tokens=True, truncation=False, padding=False)
    token_lengths.extend(len(ids) for ids in tokenized["input_ids"])
corpus["token_count"] = token_lengths
corpus["would_truncate"] = corpus["token_count"] > MAX_SEQUENCE_LENGTH
truncated = corpus.loc[corpus["would_truncate"], ["id", "token_count", "title"]].copy()
length_quantiles = corpus["token_count"].quantile([0, .25, .5, .75, .9, .95, .99, 1]).to_dict()
print("Model effective maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Tokenizer-reported maximum:", tokenizer.model_max_length)
print("Token length quantiles:", length_quantiles)
print("Documents exceeding effective limit:", len(truncated), f"({len(truncated)/len(corpus):.1%})")
display(truncated.head(25))
print("Policy: pass the complete document text to the pinned model; its configured maximum-length behavior applies. No manual truncation or chunking is introduced.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 256). Running this sequence through the model will result in indexing errors


Model effective maximum sequence length: 256
Tokenizer-reported maximum: 256
Token length quantiles: {0.0: 42.0, 0.25: 264.0, 0.5: 363.5, 0.75: 523.0, 0.9: 765.9000000000001, 0.95: 1181.699999999999, 0.99: 2012.0, 1.0: 2464.0}
Documents exceeding effective limit: 3225 (76.9%)


,id,token_count,title
1,3d2f63ae-368c-4914-b7e3-9dbc6a0885a7,609,2025-06-10-Dune: Awakening is Out Now!
2,f01be39a-f8b3-4cbb-935f-9e3a8f630720,313,2025-06-06-Dune: Awakening – 1.1.0.5 Patch Notes
3,b70adff8-3677-4a69-938d-f715977b7cb1,997,2025-06-04-Choose Your Server and Pre-Load Now!
4,e1a6e29a-500d-4729-9a3c-120bef129585,659,2025-06-05-The Head Start Has Begun! Arrakis A...
5,019e12c9-ec6a-47e3-858e-95b2aa8428c4,351,2025-06-08-Dune: Awakening – 1.1.0.9 Patch Notes
6,9a1c2f41-5fbd-4382-b291-b4ac2978200c,453,2025-06-02-Private Servers Announced
7,fdb9dd11-d80c-4b0d-aacb-e34c37e9a146,482,2025-06-07-Launch Trailer Premiered at Summer ...
8,41d88ebb-cc56-4b76-94aa-a24f34d77fae,322,2025-06-12-Dune: Awakening – 1.1.0.13 Patch Notes
9,0a34acea-1095-49e9-8885-194f17adb9a3,326,2025-06-07-Dune: Awakening – 1.1.0.7 Patch Notes
11,f18bd5dd-6d9b-4717-a927-661e73ac0051,447,2025-06-23-A message from the Creative Directo...


Policy: pass the complete document text to the pinned model; its configured maximum-length behavior applies. No manual truncation or chunking is introduced.


In [6]:
# Persist a compact audit manifest, not a second copy of the frozen corpus text.
manifest_cols = ["id", "game_name", "document_text_sha256", "document_chars", "token_count", "would_truncate"]
document_manifest_path = DATA_DIR / "chronoplay_document_manifest_v1.csv"
ingestion_report_path = MANIFEST_DIR / "chronoplay_document_ingestion_report_v1.json"
if document_manifest_path.exists() or ingestion_report_path.exists():
    raise FileExistsError("Refusing to replace existing ingestion outputs; use a new versioned filename.")
corpus[manifest_cols].to_csv(document_manifest_path, index=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

ingestion_report = {
    "status":"PASS", "dataset":"ChronoPlay-QA Claude Verified v1.1",
    "corpus_file":str(CORPUS_PATH.relative_to(HANDOFF_ROOT)), "corpus_sha256":sha256_file(CORPUS_PATH),
    "document_count":len(corpus), "unique_document_ids":int(corpus["id"].nunique()),
    "document_text":"Title: {title}\nContent: {contents}",
    "normalization":"Unicode NFC; CRLF/CR converted to LF; no other rewriting",
    "document_text_sha256":"per row in chronoplay_document_manifest_v1.csv",
    "model_id":MODEL_ID, "model_revision":MODEL_REVISION, "model_snapshot_dir":str(snapshot_dir),
    "model_config_commit":model_commit, "device":DEVICE,
    "sentence_transformers_version":sentence_transformers.__version__, "transformers_version":transformers.__version__,
    "torch_version":torch.__version__, "embedding_dimension":model.get_sentence_embedding_dimension(),
    "sentence_transformer_max_seq_length":MAX_SEQUENCE_LENGTH,
    "tokenizer_model_max_length":int(tokenizer.model_max_length),
    "truncation_policy":"No manual truncation or chunking. The pinned SentenceTransformer model applies its configured max sequence length during encoding.",
    "documents_over_limit":int(corpus["would_truncate"].sum()),
    "proportion_over_limit":float(corpus["would_truncate"].mean()),
    "affected_document_ids":truncated["id"].astype(str).tolist(),
    "token_length_quantiles":{str(k):int(v) for k,v in length_quantiles.items()},
    "outputs":{"document_manifest":str(document_manifest_path.relative_to(HANDOFF_ROOT))},
    "runtime":{"python":platform.python_version(), "platform":platform.platform()},
}
ingestion_report_path.write_text(json.dumps(ingestion_report,indent=2)+"\n",encoding="utf-8")
print("Saved:", document_manifest_path, ingestion_report_path)
print("Report SHA256:", sha256_file(ingestion_report_path))

Saved: /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/Phase2/DocumentBuilding/data/chronoplay_document_manifest_v1.csv /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/Phase2/DocumentBuilding/manifests/chronoplay_document_ingestion_report_v1.json
Report SHA256: 212e5ce5a89af5026b2fbb1a65c1fcce5021e7f048b8a6b5b50c8c7075fe4d4d


/var/folders/6x/b5gh_pcd4pbb668mxxxjshjw0000gn/T/ipykernel_8027/2459767106.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "torch_version":torch.__version__, "embedding_dimension":model.get_sentence_embedding_dimension(),
